# TPC-H: A Step-by-Step DriftBench Walkthrough

This notebook calls the same functions as `tpch.py`. By default, it only generates and inspects local files; read-only Azure data preparation can be enabled explicitly, but it does not connect to a database,
and it does not execute SQL, shell commands, Java, pgbench, or sysbench, or install or download benchmark tools.

**Benchmark boundaries:** The data step only copies local `.tbl` files that you provide; it does not run dbgen or download SF10 by default. Point `SOURCE_DIR` to a small TPC-H data directory (all inputs must total no more than 64 MiB) and explicitly set `SOURCE_SCALE_FACTOR`; the input scale is not inferred from function defaults, and the declaration is not content validation. The example inspects lineitem. Queries are Python qgen-style SQL instances, not official qgen output, and do not represent TPC certification.

`PASS` means that the step has concrete evidence; `NEEDS_INPUT` requires local input; `UNAVAILABLE` means the selected package lacks the API or dependency;
`NOT_SUPPORTED` means that the capability does not exist. Any incomplete step makes the overall result `PARTIAL`, not "all benchmark checks succeeded."
Invalid parameters and genuine runtime errors raise exceptions; they are not converted into skips or successes.

## 0. Prepare the environment manually

Copy this entire directory, including `_walkthrough.py` and all companion files, and launch the notebook from this directory.
For PyPI mode, install `driftbench-db` and Jupyter in an isolated virtual environment, and do not start the kernel from the repository root.
For source mode, explicitly install this checkout first (`pip install -e .`), then change the target below to `checkout`.
These companion documents are not guaranteed to be included in an installed package. See the [README](README.md) for detailed commands, status meanings, and input layouts.


In [ ]:
from pathlib import Path
from tpch import create

PACKAGE_TARGET = "installed"
OUTPUT_DIR = Path.home() / "DriftBenchWalkthroughs"
SEED = 42
SAMPLE_SIZE = 200
RECORD_COUNT = 200
SOURCE_DIR = None
SOURCE_SCALE_FACTOR = None
PARAMETERS_DIR = None
DRIVER_CONFIG = None
AZURE_DATASET_ID = None
DATASET_CACHE_DIR = Path.home() / "DriftBenchDatasets"
AZURE_CREDENTIAL_ENV_FILE = None


## 1. Confirm exactly which package is under test

Check the distribution version, module version, and actual imported file. A version number alone does not prove identical functionality.
`installed` rejects source-tree shadowing and editable installs; do not silently switch to source mode to make the check pass.
Each call to create makes a new persistent output subdirectory; earlier runs are neither overwritten nor deleted automatically.


In [ ]:
demo = create(
    output_dir=OUTPUT_DIR, package_target=PACKAGE_TARGET,
    seed=SEED, sample_size=SAMPLE_SIZE, record_count=RECORD_COUNT,
    source_dir=SOURCE_DIR, source_scale_factor=SOURCE_SCALE_FACTOR,
    parameters_dir=PARAMETERS_DIR, driver_config=DRIVER_CONFIG,
)
demo.identity


## 1a. Optional: Prepare and cache real SF0.01 data from Azure

By default, `AZURE_DATASET_ID = None`, so Azure is not accessed at all. To read data that you are authorized to access, change the settings cell to:

```python
AZURE_DATASET_ID = "tpch/data/immutable-dataset-v1/72764dad47622f7d5e0387b6faf623f2a4257648533b6f08040adcc6393cba2c"
```

This is approximately 10.6 MB across all eight complete SF0.01 tables, not a sample of SF10. Keep `SOURCE_DIR` and `SOURCE_SCALE_FACTOR` as `None`;
the source and scale are read from the explicitly selected catalog entry. `PACKAGE_TARGET` must still reflect the package you actually selected.
The current development version provides `catalog.materialize`; an installed package without this API reports `UNAVAILABLE` instead of switching to source code.

The first run uses an existing Azure identity for a read-only download and validation, producing `downloaded`. Later runs first check every cached file,
its size and SHA-256, the commit/provenance metadata, and source binding. A successful check produces `hit` without initializing an Azure client or reading credentials.
Cache corruption raises an error; it is not overwritten or downloaded again automatically. A valid cache does not prove that the remote data still exists, access is still valid, or the catalog has been refreshed in real time.
The cache is stored in a separate directory; data conversion and drift write only to another run directory. Do not commit or publish these private inputs.

Only the explicit preparation step below may access Azure; subsequent steps still prohibit network access, external drivers, and database execution.
It does not install dependencies, sign in, or change permissions. If the optional Azure dependency or an identity is unavailable, prepare the environment manually by following the README first.


In [ ]:
if AZURE_DATASET_ID is not None:
    azure_source = demo.prepare_azure_data(
        AZURE_DATASET_ID, cache_dir=DATASET_CACHE_DIR,
        credential_env_file=AZURE_CREDENTIAL_ENV_FILE,
    )
    print({key: azure_source.get(key) for key in (
        "status", "cache_outcome", "payload_dir", "file_count",
        "payload_bytes", "downloaded_payload_bytes", "binding_sha256",
    )})
else:
    print("Azure preparation not requested; using the explicit local-input workflow.")

## 2. Generate or import baseline files

This calls the real public entry point `driftbench.data.tpch.data(...).generate(..., force=False)`.
The fixed small-scale parameters are `{'mode': 'copy'}`; YCSB uses the record_count configured above.
TPC-H, SSB, and LDBC only read explicitly provided local inputs; missing input never falls back to fabricated data.
source_dir and the output directory must be separate, and inputs are limited to 64 MiB. Do not use SF10 as the default tutorial input.


In [ ]:
demo.generate_data()

## 3. Load and inspect files

`.tbl` and `.dat` files are converted with the public `GenerationResult.as_csv()` method; existing CSV files are read using their actual delimiter.
The DataFrame is a client-side check performed with Pandas; it does not mean that DriftBench loaded the data into a database.
This reports the selected table's actual row count, column names, and path; other generated tables remain in the output directory.


In [ ]:
demo.load_data()

In [ ]:
demo.preview()

## 4. Change the row count

This calls `GenerationResult.drift(table, "vary_cardinality", scale=..., seed=..., output_path=...)`.
The expected row count is `int(original_row_count * scale)`. This operation regenerates data for the existing columns; it does not simply delete old rows.
Primary keys, foreign keys, join results, and graph integrity are not guaranteed. The original file's SHA-256 must remain unchanged.
You can modify scale below. The tutorial range is `(0, 2]`; a value too small to generate one row produces an explicit message.


In [ ]:
demo.drift_data(scale=0.5)

## 5. Generate real query or workload artifacts

The public entry point is `driftbench.data.tpch.queries(...)`. The current profile parameters are:

```python
[{'query_ids': [1, 6], 'queries_per_template': 2, 'shuffle': False}]
```

The results distinguish SQL instances, SQL templates, query IDs, operation names, and native-configuration references. Duplicate SQL collections and table-creation SQL
are not counted as new queries. LDBC does not display the configuration contents. This step only reads text; it executes none of the artifacts.


In [ ]:
query_result = demo.generate_queries()
{key: value for key, value in query_result.items() if key != "templates"}

In [ ]:
[(item["id"], item["kind"]) for item in demo.templates[:20]]

## 6. Produce a baseline stream on demand

This uses `driftbench.api.QueryTemplate` and `execute_query_template_mix_spec` instead of reimplementing the sampler in the tutorial.
`SAMPLE_SIZE` is the stream length, which is distinct from a data table's row count. Default YCSB and BenchBase weights come from generated artifacts;
the other examples explicitly use an equal-weight teaching baseline. You can set `BASELINE_WEIGHTS` to a dictionary covering every template ID.

The report shows normalized weights together with actual sample counts and proportions, and checks length, membership, zero weights, and reproducibility with the same seed.
Weights are not exact count quotas; the stream does not guarantee real-time QPS, arrival times, query costs, selectivities, or database replay.
If the package lacks this public API, the step records UNAVAILABLE instead of substituting a private implementation or a custom random sequence.


In [ ]:
BASELINE_WEIGHTS = None
demo.query_stream(weights=BASELINE_WEIGHTS)

## 7. Change query or operation frequencies

By default, the first template ID receives a target weight of 0.8, and the remaining templates share 0.2.
You can also provide complete `TARGET_WEIGHTS`; for example, to retain only one ID, explicitly assign 0 to every other ID.
This changes the mix proportions, not SQL predicates. Both the original baseline and the changed sequence are saved in JSON.


In [ ]:
TARGET_WEIGHTS = None
demo.change_mix(weights=TARGET_WEIGHTS)

## 8. Change public parameters or the profile

This example supports changes to `('query_ids', 'queries_per_template', 'seed', 'shuffle', 'scale')`.
The default change is `{'query_ids': [3, 6], 'queries_per_template': 3}`. An empty collection means that this kind of rewriting is unavailable,
and the step records NOT_SUPPORTED accordingly. Edit the parameter dictionary below instead of modifying generated files.

The old and new artifacts must differ, while the old query-file hashes remain unchanged. TPC-C Skew changes comments;
driver parameters such as rate, duration, and distribution change configuration but do not prove that the requested workload was actually executed.


In [ ]:
CHANGE_OPTIONS = {'query_ids': [3, 6], 'queries_per_template': 3}
demo.change_query_parameters(**CHANGE_OPTIONS)

## 9. Review the summary and persistent evidence

Review the actual status of each step; do not treat PARTIAL as a complete pass. The JSON report and all local artifacts remain in this run's exclusive directory.
Committed notebooks contain no execution output. Do not commit results containing real local inputs or credentials to Git.
Each rerun creates a new directory. Delete a specific run directory manually only after confirming that it is no longer needed; never clean the entire output root.


In [ ]:
report = demo.report()
[(name, value["status"], value["reason"]) for name, value in report["steps"].items()]

In [ ]:
{"status": report["status"], "report_file": report["report_file"]}